In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from prophet import Prophet

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from itertools import product

In [ ]:
df = pd.read_parquet(r"files\fact_ventas_predict.parquet")


In [67]:
df["flag"] = np.where(df["y"]==0,1,0)
df["flag2"] = np.where((df["ds"].dt.month==1)&(df["ds"].dt.day==1),1,0)
df["flag3"] = df["flag"].astype(str)+df["flag2"].astype(str)
df[df["flag3"]=="10"]

df1 = df[df["flag3"]!="10"].drop(columns=["flag", "flag2", "flag3"])

In [80]:
fecha_fin = df1["ds"].max() - pd.Timedelta(days=30)

# 2. Filtrar el DataFrame para la ventana de los últimos 30 días (Vectorizado)
df_ultimos_30 = df1[df1["ds"] >= fecha_fin]

# 3. Contar días únicos de venta por 'llave' en esa ventana (Eficiencia O(N))
# Usamos nunique() por si una 'llave' tiene múltiples registros en un mismo día
conteo_dias_venta = df_ultimos_30.groupby("llave")["ds"].nunique()

# 4. Filtrar las llaves que registren por lo menos 20 días con actividad
llaves_vigentes = conteo_dias_venta[conteo_dias_venta >= 20].index

# 5. Aplicar indexación booleana sobre el DataFrame original para obtener df2
df2 = df1[df1["llave"].isin(llaves_vigentes)]

# OPCIONAL: Si necesitas la lista de NO vigentes por consistencia con tu código previo
# Usamos operaciones de conjuntos (sets) que son computacionalmente más rápidas que listas
todas_las_llaves = set(df1["llave"].unique())
lista_llave_vigente = list(llaves_vigentes)
lista_llave_No_vigente = list(todas_las_llaves - set(llaves_vigentes))

In [81]:
df2_grupo = df2.groupby("llave").size().reset_index(name="cant")
lista_llave_vigente_historia_120 = df2_grupo[df2_grupo["cant"]>120]["llave"].unique().tolist()

df3 = df2[df2["llave"].isin(lista_llave_vigente_historia_120)]

In [85]:
df_venta_cero = df3[(df3["ds"].dt.month==1)&(df3["ds"].dt.day==1)].groupby("llave")["y"].sum().reset_index()
list_local_open = df_venta_cero[df_venta_cero["y"]>500]["llave"].unique().tolist()
list_llaves = df3["llave"].unique().tolist()




In [92]:
df3[df3["llave"]=="V116-D1A01"].to_excel("revisar8.xlsx")

In [94]:
def entrenar_evaluar_ensamble(df, llave, fecha):
    # ==============================================================================
    # 1. ESQUEMA DE CONTINGENCIA POR DEFECTO (FALLBACK)
    # ==============================================================================
    df_fallback = pd.DataFrame(
        {
            "llave": [llave],
            "fecha": [pd.to_datetime(fecha)],
            "prophet train rmse": [np.nan],
            "prophet test rmse": [np.nan],
            "prophet train r2": [np.nan],
            "prophet test r2": [np.nan],
            "prophet error mes": [np.nan],
            "ensamble train rmse": [np.nan],
            "ensamble test rmse": [np.nan],
            "ensamble train r2": [np.nan],
            "ensamble test r2": [np.nan],
            "ensamble error mes": [np.nan],
        }
    )

    # Filtrados iniciales de series temporales
    dff = df[df["llave"] == llave]
    dff_01 = dff[dff["ds"] < fecha]

    # Early Exit si la serie temporal no tiene suficientes datos históricos o de prueba
    if dff_01.empty or dff[dff["ds"] >= fecha].empty:
        return df_fallback

    try:
        # ==============================================================================
        # 2. ENTRENAMIENTO E INFERENCIA CON PROPHET
        # ==============================================================================
        dff_01_prophet = dff_01[["ds", "y"]].copy()

        model_prophet = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            seasonality_mode="additive",
            seasonality_prior_scale=10,
            changepoint_prior_scale=0.05,
        )
        model_prophet.fit(dff_01_prophet)

        future = model_prophet.make_future_dataframe(periods=30, freq="D")
        forecast = model_prophet.predict(future)

        # Merge vectorial eliminando columnas redundantes
        forecast1 = forecast[
            ["ds", "trend", "weekly", "yearly", "yhat"]
        ].merge(
            dff.drop(columns=["DivArea", "Codigo Local", "llave"]),
            on="ds",
            how="left",
        )
        forecast1 = forecast1.fillna(0)

        # ==============================================================================
        # 3. MODELADO DE ENSAMBLE (REGRESIÓN LINEAL)
        # ==============================================================================
        X_train = forecast1[forecast1["ds"] < fecha].drop(
            columns=["y", "ds", "yhat"]
        )
        y_train = forecast1[forecast1["ds"] < fecha]["y"]

        X_test = forecast1[forecast1["ds"] >= fecha].drop(
            columns=["y", "ds", "yhat"]
        )
        y_test = forecast1[forecast1["ds"] >= fecha]["y"]

        # Control estructural obligatorio para scikit-learn
        if X_train.shape[0] == 0 or X_test.shape[0] == 0:
            return df_fallback

        model_lr = LinearRegression(n_jobs=-1)
        model_lr.fit(X_train, y_train)

        # Inferencia y unificación de predicciones
        y_pred_train = model_lr.predict(X_train)
        y_pred_test = model_lr.predict(X_test)
        forecast1["yhat2"] = np.concatenate([y_pred_train, y_pred_test])

        # ==============================================================================
        # 4. LÓGICA DE RESTRICCIÓN DE NEGOCIO (LOCALES CERRADOS)
        # ==============================================================================
        if llave not in list_local_open:
            forecast1["yhat2"] = np.where(
                (forecast1["ds"].dt.month == 1) & (forecast1["ds"].dt.day == 1),
                0,
                forecast1["yhat2"],
            )

        # ==============================================================================
        # 5. CÁLCULO DE MÈTRICAS (PROPHET VS ENSAMBLE)
        # ==============================================================================
        y_train_real = forecast1[forecast1["ds"] < fecha]["y"]
        y_test_real = forecast1[forecast1["ds"] >= fecha]["y"]

        yhat_train = forecast1[forecast1["ds"] < fecha]["yhat"]
        yhat_test = forecast1[forecast1["ds"] >= fecha]["yhat"]

        yhat2_train = forecast1[forecast1["ds"] < fecha]["yhat2"]
        yhat2_test = forecast1[forecast1["ds"] >= fecha]["yhat2"]

        sum_y_test = y_test_real.sum()
        prophet_err_mes = (
            yhat_test.sum() / sum_y_test if sum_y_test != 0 else np.nan
        )
        ensamble_err_mes = (
            yhat2_test.sum() / sum_y_test if sum_y_test != 0 else np.nan
        )

        # ==============================================================================
        # 6. CONSTRUCCIÓN DEL DATAFRAME DE SALIDA (MÉTRICAS COLECTADAS)
        # ==============================================================================
        df_metrics = pd.DataFrame(
            {
                "llave": [llave],
                "fecha": [pd.to_datetime(fecha)],
                "prophet train rmse": [
                    np.sqrt(mean_squared_error(y_train_real, yhat_train))
                ],
                "prophet test rmse": [
                    np.sqrt(mean_squared_error(y_test_real, yhat_test))
                ],
                "prophet train r2": [r2_score(y_train_real, yhat_train)],
                "prophet test r2": [r2_score(y_test_real, yhat_test)],
                "prophet error mes": [prophet_err_mes],
                "ensamble train rmse": [
                    np.sqrt(mean_squared_error(y_train_real, yhat2_train))
                ],
                "ensamble test rmse": [
                    np.sqrt(mean_squared_error(y_test_real, yhat2_test))
                ],
                "ensamble train r2": [r2_score(y_train_real, yhat2_train)],
                "ensamble test r2": [r2_score(y_test_real, yhat2_test)],
                "ensamble error mes": [ensamble_err_mes],
            }
        )
        return df_metrics

    except Exception as e:
        print(f"Error procesando llave {llave} en periodo {fecha}: {str(e)}")
        return df_fallback




In [95]:
list_llaves = df3["llave"].unique().tolist()
p1, p2, p3, p4, p5 = [list(x) for x in np.array_split(list_llaves, 5)]

In [96]:

list_period = ["2026-01-01", "2026-02-01", "2026-03-01", "2026-04-01"]

resultados_list = []

for llave in p1:
    print(llave)
    for periodo in list_period:

        df_resultado_individual = entrenar_evaluar_ensamble( df3,llave, periodo )
        resultados_list.append(df_resultado_individual)

df_resultados_total_p1 = pd.concat(resultados_list, ignore_index=True)

22:14:55 - cmdstanpy - INFO - Chain [1] start processing
22:14:55 - cmdstanpy - INFO - Chain [1] done processing


1198-D1A01


22:14:55 - cmdstanpy - INFO - Chain [1] start processing
22:14:55 - cmdstanpy - INFO - Chain [1] done processing
22:14:55 - cmdstanpy - INFO - Chain [1] start processing
22:14:55 - cmdstanpy - INFO - Chain [1] done processing
22:14:55 - cmdstanpy - INFO - Chain [1] start processing
22:14:55 - cmdstanpy - INFO - Chain [1] done processing
22:14:56 - cmdstanpy - INFO - Chain [1] start processing
22:14:56 - cmdstanpy - INFO - Chain [1] done processing


1199-D1A01


22:14:56 - cmdstanpy - INFO - Chain [1] start processing
22:14:56 - cmdstanpy - INFO - Chain [1] done processing
22:14:56 - cmdstanpy - INFO - Chain [1] start processing
22:14:56 - cmdstanpy - INFO - Chain [1] done processing
22:14:56 - cmdstanpy - INFO - Chain [1] start processing
22:14:56 - cmdstanpy - INFO - Chain [1] done processing
22:14:57 - cmdstanpy - INFO - Chain [1] start processing
22:14:57 - cmdstanpy - INFO - Chain [1] done processing


...

22:20:57 - cmdstanpy - INFO - Chain [1] start processing
22:20:57 - cmdstanpy - INFO - Chain [1] done processing
22:20:57 - cmdstanpy - INFO - Chain [1] start processing
22:20:57 - cmdstanpy - INFO - Chain [1] done processing
22:20:57 - cmdstanpy - INFO - Chain [1] start processing
22:20:57 - cmdstanpy - INFO - Chain [1] done processing
22:20:58 - cmdstanpy - INFO - Chain [1] start processing
22:20:58 - cmdstanpy - INFO - Chain [1] done processing


V093-D1A03


22:20:58 - cmdstanpy - INFO - Chain [1] start processing
22:20:58 - cmdstanpy - INFO - Chain [1] done processing
22:20:58 - cmdstanpy - INFO - Chain [1] start processing
22:20:58 - cmdstanpy - INFO - Chain [1] done processing
22:20:58 - cmdstanpy - INFO - Chain [1] start processing
22:20:59 - cmdstanpy - INFO - Chain [1] done processing
22:20:59 - cmdstanpy - INFO - Chain [1] start processing
22:20:59 - cmdstanpy - INFO - Chain [1] done processing


V094-D1A03


22:20:59 - cmdstanpy - INFO - Chain [1] start processing
22:20:59 - cmdstanpy - INFO - Chain [1] done processing
22:20:59 - cmdstanpy - INFO - Chain [1] start processing
22:20:59 - cmdstanpy - INFO - Chain [1] done processing
22:21:00 - cmdstanpy - INFO - Chain [1] start processing
22:21:00 - cmdstanpy - INFO - Chain [1] done processing


In [97]:
resultados_list = []

for llave in p2:
    print(llave)
    for periodo in list_period:
        print(periodo)
        df_resultado_individual = entrenar_evaluar_ensamble( df3,llave, periodo )
        resultados_list.append(df_resultado_individual)

df_resultados_total_p2 = pd.concat(resultados_list, ignore_index=True)

22:21:00 - cmdstanpy - INFO - Chain [1] start processing
22:21:00 - cmdstanpy - INFO - Chain [1] done processing


V095-D1A03
2026-01-01


22:21:00 - cmdstanpy - INFO - Chain [1] start processing
22:21:00 - cmdstanpy - INFO - Chain [1] done processing


2026-02-01


22:21:00 - cmdstanpy - INFO - Chain [1] start processing
22:21:00 - cmdstanpy - INFO - Chain [1] done processing


...

22:26:57 - cmdstanpy - INFO - Chain [1] start processing


2026-03-01


22:26:57 - cmdstanpy - INFO - Chain [1] done processing
22:26:57 - cmdstanpy - INFO - Chain [1] start processing


2026-04-01


22:26:57 - cmdstanpy - INFO - Chain [1] done processing


In [98]:
resultados_list = []

for llave in p3:
    print(llave)
    for periodo in list_period:
        print(periodo)
        df_resultado_individual = entrenar_evaluar_ensamble( df3,llave, periodo )
        resultados_list.append(df_resultado_individual)

df_resultados_total_p3 = pd.concat(resultados_list, ignore_index=True)

22:26:58 - cmdstanpy - INFO - Chain [1] start processing


SO04-D2A06
2026-01-01


22:26:58 - cmdstanpy - INFO - Chain [1] done processing
22:26:58 - cmdstanpy - INFO - Chain [1] start processing
22:26:58 - cmdstanpy - INFO - Chain [1] done processing


2026-02-01


22:26:58 - cmdstanpy - INFO - Chain [1] start processing


...

2026-02-01


22:33:01 - cmdstanpy - INFO - Chain [1] start processing
22:33:01 - cmdstanpy - INFO - Chain [1] done processing


2026-03-01


22:33:02 - cmdstanpy - INFO - Chain [1] start processing
22:33:02 - cmdstanpy - INFO - Chain [1] done processing


2026-04-01


In [99]:
resultados_list = []

for llave in p5:
    print(llave)
    for periodo in list_period:
        df_resultado_individual = entrenar_evaluar_ensamble( df3,llave, periodo )
        resultados_list.append(df_resultado_individual)

df_resultados_total_p5 = pd.concat(resultados_list, ignore_index=True)

22:33:02 - cmdstanpy - INFO - Chain [1] start processing


P059-D3A13


22:33:02 - cmdstanpy - INFO - Chain [1] done processing
22:33:02 - cmdstanpy - INFO - Chain [1] start processing
22:33:02 - cmdstanpy - INFO - Chain [1] done processing
22:33:03 - cmdstanpy - INFO - Chain [1] start processing
22:33:03 - cmdstanpy - INFO - Chain [1] done processing
22:33:03 - cmdstanpy - INFO - Chain [1] start processing
22:33:03 - cmdstanpy - INFO - Chain [1] done processing
22:33:03 - cmdstanpy - INFO - Chain [1] start processing
22:33:03 - cmdstanpy - INFO - Chain [1] done processing


P061-D3A13


22:33:03 - cmdstanpy - INFO - Chain [1] start processing
22:33:03 - cmdstanpy - INFO - Chain [1] done processing
22:33:04 - cmdstanpy - INFO - Chain [1] start processing
22:33:04 - cmdstanpy - INFO - Chain [1] done processing
22:33:04 - cmdstanpy - INFO - Chain [1] start processing
22:33:04 - cmdstanpy - INFO - Chain [1] done processing
22:33:04 - cmdstanpy - INFO - Chain [1] start processing
22:33:04 - cmdstanpy - INFO - Chain [1] done processing


...

22:39:06 - cmdstanpy - INFO - Chain [1] start processing
22:39:06 - cmdstanpy - INFO - Chain [1] done processing
22:39:06 - cmdstanpy - INFO - Chain [1] start processing
22:39:06 - cmdstanpy - INFO - Chain [1] done processing
22:39:07 - cmdstanpy - INFO - Chain [1] start processing
22:39:07 - cmdstanpy - INFO - Chain [1] done processing
22:39:07 - cmdstanpy - INFO - Chain [1] start processing
22:39:07 - cmdstanpy - INFO - Chain [1] done processing


X209-D4A14


22:39:07 - cmdstanpy - INFO - Chain [1] start processing
22:39:07 - cmdstanpy - INFO - Chain [1] done processing
22:39:07 - cmdstanpy - INFO - Chain [1] start processing
22:39:07 - cmdstanpy - INFO - Chain [1] done processing
22:39:08 - cmdstanpy - INFO - Chain [1] start processing
22:39:08 - cmdstanpy - INFO - Chain [1] done processing
22:39:08 - cmdstanpy - INFO - Chain [1] start processing
22:39:08 - cmdstanpy - INFO - Chain [1] done processing


X210-D4A14


22:39:08 - cmdstanpy - INFO - Chain [1] start processing
22:39:08 - cmdstanpy - INFO - Chain [1] done processing
22:39:09 - cmdstanpy - INFO - Chain [1] start processing
22:39:09 - cmdstanpy - INFO - Chain [1] done processing
22:39:09 - cmdstanpy - INFO - Chain [1] start processing
22:39:09 - cmdstanpy - INFO - Chain [1] done processing


In [100]:
resultados_list = []

for llave in p4:
    print(llave)
    for periodo in list_period:
        print(periodo)
        df_resultado_individual = entrenar_evaluar_ensamble( df3,llave, periodo )
        resultados_list.append(df_resultado_individual)

df_resultados_total_p4 = pd.concat(resultados_list, ignore_index=True)

22:39:09 - cmdstanpy - INFO - Chain [1] start processing
22:39:09 - cmdstanpy - INFO - Chain [1] done processing


P465-D2A09
2026-01-01


22:39:09 - cmdstanpy - INFO - Chain [1] start processing
22:39:10 - cmdstanpy - INFO - Chain [1] done processing


2026-02-01


22:39:10 - cmdstanpy - INFO - Chain [1] start processing
22:39:10 - cmdstanpy - INFO - Chain [1] done processing


...

22:45:12 - cmdstanpy - INFO - Chain [1] start processing
22:45:12 - cmdstanpy - INFO - Chain [1] done processing


2026-03-01


22:45:12 - cmdstanpy - INFO - Chain [1] start processing


2026-04-01


22:45:12 - cmdstanpy - INFO - Chain [1] done processing


In [101]:
dffff = pd.concat([df_resultados_total_p1, df_resultados_total_p2, df_resultados_total_p3, df_resultados_total_p4, df_resultados_total_p5])

In [104]:
dffff.to_excel("revisar9.xlsx")